# 02 — The Agent Loop: Weather Forecaster

**Module 1 of the workshop.** The "wow" opener — a single agent autonomously chains two real HTTP calls with no hand-written control flow.


## Problem

Turning an LLM (stateless text-in, text-out) into something that can *act* — call functions, return structured data, chain steps — without you hand-writing that control flow.

Concretely here: answering "what's the weather in Seattle?" needs two sequential API calls (grid lookup, then forecast fetch), and the second call's URL only exists after the first one returns. A plain LLM can't do that. An agent with tools can.


## Concept

`Agent(model=model, tools=tools)`, then `result = agent("do something")`. Three moving parts: model, tools, and the loop that ties them together.

**What happens internally:** Prompt → Agent → Model → (tool call? → Tool → Tool result → Model again) → final answer → User. This *is* the agent loop — the model decides, unprompted, whether it needs another tool call or is ready to answer.

**When would you NOT use an agent?** When the task is a fixed, known sequence of steps with no reasoning or branching needed — that's just a function. Agents earn their cost when the *next step depends on what just happened* (exactly the case here: the forecast URL only exists after the grid-lookup response comes back).


## Architecture

```
"What's the weather in Seattle?"
              │
              ▼
        ┌───────────┐
        │  Agent    │◀──────────────────────┐
        │ (model)   │                        │
        └─────┬─────┘                        │
              │ decides: need a tool call     │
              ▼                               │
     ┌──────────────────┐                     │
     │ http_request /    │                    │
     │ web_fetch (tool)  │                    │
     └────────┬──────────┘                    │
              │ GET api.weather.gov/points/... │
              ▼                                │
     grid info (incl. forecast URL) ───────────┘
                                                (model reasons: need forecast too)
              ┌─────────────────────────────────┘
              ▼
     ┌──────────────────┐
     │ http_request      │
     │ (tool, 2nd call)  │
     └────────┬──────────┘
              │ GET <forecast URL from step above>
              ▼
     forecast data ──────────▶ Agent formats ──────────▶ final answer to user
```

Two HTTP calls, unprompted — the agent decides to chain them because the system prompt tells it *what* to do (grid lookup, then forecast), and the loop lets it *act* on that plan one tool call at a time.


## Step 1 — Resolve the model

Ollama primary, Bedrock fallback — same resolver as every script in this workshop.


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from model_provider import get_model
from strands import Agent
from strands.vended_tools import http_request, web_fetch

model = get_model()
print(f"Using: {type(model).__name__}")


Using: OllamaModel


## Step 2 — Write the system prompt

This is where the *plan* lives: tell the agent the two-step API sequence (grid lookup, then forecast), not just "answer weather questions." The agent loop only chains calls correctly if it knows there are two calls to make.


In [2]:
WEATHER_SYSTEM_PROMPT = """You are a weather assistant with HTTP capabilities. You can:
1. Make HTTP requests to the National Weather Service API
2. Process and display weather forecast data
3. Provide weather information for locations in the United States

When retrieving weather information:
1. First get grid info via https://api.weather.gov/points/{latitude},{longitude}
2. Then use the returned forecast URL to get the actual forecast

Format weather data in a human-readable way, highlight temperature, precipitation,
and alerts, handle errors gracefully, and explain conditions clearly."""


## Step 3 — Build the agent with vended tools

`http_request` and `web_fetch` are **vended tools** — pre-built capabilities Strands ships, not something you write yourself. `agent_id` names this agent (useful once you have more than one running).


In [3]:
weather_agent = Agent(
    model=model,
    agent_id="weather_agent",
    system_prompt=WEATHER_SYSTEM_PROMPT,
    tools=[http_request, web_fetch]
)


## Step 4 — Run it and watch the loop

The agent will visibly reason through both HTTP calls without you prompting each one — that's the loop from the architecture diagram, live.


In [4]:
result = weather_agent("What's the weather like in Seattle? (lat 47.6062, lon -122.3321)")


Tool #1: http_request

Tool #2: http_request
# 🌤️ 7-Day Weather Forecast (September 26-30, 2026)

## **Saturday Evening**
| Metric | Value |
|--------|-------|
| Temp Range | High: 63°F / Low: 50°F |
| Condition | Mostly Sunny |
| Wind | North at 6 mph |
| Rain Chance | — |

---

## **Sunday** 
- **Morning**: High ~63°F, Mostly Sunny (N wind 6 mph)  
- **Night**: Low ~50°F, Partly Cloudy (ENE wind 3 mph), 6% rain  

---

## **Monday**
| Metric | Value |
|--------|-------|
| Morning | High ~63°F, Mostly Cloudy (S wind 3 mph) |
| Night | Low ~53°F, 20-40% Rain Chance (S wind 2-9 mph) |

---

## **Tuesday**
| Metric | Value |
|--------|-------|
| Day | High ~62°F, 44% Rain Chance (S wind 7-10 mph) |
| Night | Low ~53°F, 44% Rain Chance (S wind 3-8 mph) |

---

## **Wednesday**
| Metric | Value |
|--------|-------|
| Day | High ~63°F, Mostly Cloudy (NNW wind 5 mph) |
| Night | Low ~52°F, 27% Rain Chance (NE wind 3 mph) |

---

## **Thursday**
| Metric | Value |
|--------|-------|
| Day | 

## Failure mode to know about

An agent with no tools and a vague prompt will confidently make things up rather than admit it doesn't know — the loop still runs, it just never branches into a tool call. Tools aren't optional plumbing, they're what keeps the loop honest. If this agent had no `http_request` tool, it would still answer the weather question — just with hallucinated data instead of the real forecast.
